# 5.12 · 线性 & 二次判别分析 / LDA & QDA

> **课程定位 / Where this fits**
> 朴素贝叶斯(5.4)是生成式但假设特征独立。LDA/QDA 也是生成式, 但用**多元高斯**对每类建模(允许特征相关)。LDA 假设各类协方差相同 → **线性边界**; QDA 各类协方差不同 → **二次边界**。LDA 还能做**有监督降维**(对比 Part 7 的 PCA)。
> LDA/QDA model each class as a multivariate Gaussian. Shared covariance → linear boundary (LDA); per-class covariance → quadratic (QDA). LDA also does supervised dimensionality reduction.

> 💡 **面试相关 / Interview-relevant**
> - "LDA 与 QDA 的区别 / 边界为何线/二次" ★★★★★
> - "LDA 与 PCA 区别" ★★★★★（有监督 vs 无监督）
> - "LDA 与逻辑回归区别" ★★★★（生成 vs 判别, 假设）
> - "LDA 的假设 / 何时失效" ★★★★
> - "QDA 参数多→何时过拟合" ★★★

---

## 学习目标 / Learning Objectives
1. 从贝叶斯 + 高斯似然推出 LDA/QDA 判别函数。
2. 看清协方差假设如何决定边界形状。
3. LDA 作为**有监督降维**(vs PCA)。
4. LDA / QDA / 逻辑回归 / NB 的关系网。

## 目录 / TOC
1. [高斯生成模型 → 判别函数 ⭐](#1)
2. [LDA 线性 vs QDA 二次 ⭐](#2)
3. [🍷 数据: Wine + 边界可视化](#3)
4. [LDA 有监督降维 vs PCA ⭐](#4)
5. [关系网: LDA/QDA/LR/NB](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 高斯生成模型 → 判别函数 ⭐ / Gaussian Generative Model

假设每类 $k$ 的特征服从**多元高斯** $\mathbf{x}\mid y=k \sim \mathcal{N}(\boldsymbol\mu_k, \boldsymbol\Sigma_k)$, 类先验 $\pi_k$。由贝叶斯(2.8):
$$\Pr(y=k\mid\mathbf{x}) \propto \pi_k\,\mathcal{N}(\mathbf{x};\boldsymbol\mu_k,\boldsymbol\Sigma_k)$$

取对数(扔掉与 $k$ 无关项), 得**判别函数**:
$$\delta_k(\mathbf{x}) = -\tfrac12\log|\boldsymbol\Sigma_k| - \tfrac12(\mathbf{x}-\boldsymbol\mu_k)^\top\boldsymbol\Sigma_k^{-1}(\mathbf{x}-\boldsymbol\mu_k) + \log\pi_k$$
预测 = $\arg\max_k \delta_k$。那个二次型 $(\mathbf{x}-\boldsymbol\mu_k)^\top\boldsymbol\Sigma_k^{-1}(\mathbf{x}-\boldsymbol\mu_k)$ 正是**马氏距离**——LDA/QDA = 分到"马氏距离最近(且先验加权)"的类。


<a id="2"></a>
## 2. LDA 线性 vs QDA 二次 ⭐ / Linear vs Quadratic

关键在协方差假设:

- **QDA**: 每类有**自己的** $\boldsymbol\Sigma_k$。$\delta_k$ 含 $\mathbf{x}^\top\boldsymbol\Sigma_k^{-1}\mathbf{x}$, 是 $\mathbf{x}$ 的**二次函数** → 决策边界**二次曲线/曲面**。参数多($K$ 个协方差矩阵), 数据少易过拟合。

- **LDA**: 假设**所有类共享** $\boldsymbol\Sigma$。展开 $\delta_k$ 时二次项 $\mathbf{x}^\top\boldsymbol\Sigma^{-1}\mathbf{x}$ **与 $k$ 无关**被消掉 → $\delta_k$ 是 $\mathbf{x}$ 的**线性函数** → 边界**线性**(超平面)。参数少、更稳。

一句话: **LDA 是 QDA 在"等协方差"假设下的简化**, 用偏差换方差。


<a id="3"></a>
## 3. 数据: Wine + 边界可视化 / Wine Dataset

**Wine**: sklearn 内置, 178 瓶意大利葡萄酒, 13 个化学成分特征(酒精、酸度、酚类等), 3 个产地品种。各类大致服从高斯, 适合 LDA/QDA。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
sns.set_theme(style="whitegrid")

wine = load_wine()
X, y = wine.data, wine.target
print(f"Wine: {X.shape}, 3 类各 {np.bincount(y)}")
print("特征示例:", list(wine.feature_names[:5]), "...")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)

lda = LinearDiscriminantAnalysis().fit(Xtr, y_tr)
qda = QuadraticDiscriminantAnalysis().fit(Xtr, y_tr)
print(f"LDA test 准确率: {lda.score(Xte, y_te):.3f}")
print(f"QDA test 准确率: {qda.score(Xte, y_te):.3f}")


In [ ]:
# 2D 边界: LDA 直线 vs QDA 曲线 / boundaries on 2 features
X2 = Xtr[:, [0, 6]]   # alcohol, flavanoids
lda2 = LinearDiscriminantAnalysis().fit(X2, y_tr)
qda2 = QuadraticDiscriminantAnalysis().fit(X2, y_tr)
xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-1, X2[:,0].max()+1, 300),
                     np.linspace(X2[:,1].min()-1, X2[:,1].max()+1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, m, name in [(axes[0], lda2, "LDA → 直线边界"), (axes[1], qda2, "QDA → 二次曲线边界")]:
    Z = m.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
    ax.scatter(X2[:,0], X2[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=25)
    ax.set_xlabel("alcohol (std)"); ax.set_ylabel("flavanoids (std)"); ax.set_title(name)
plt.tight_layout(); plt.show()
print("LDA(共享协方差)边界是直线; QDA(各类协方差)边界可弯曲")


<a id="4"></a>
## 4. LDA 有监督降维 vs PCA ⭐ / LDA as Supervised DR

LDA 还能降维: 它找**最大化类间方差 / 类内方差**的方向(最多 $K-1$ 维)。和 PCA(Part 7)对比:
- **PCA**: 无监督, 找方差最大的方向(不看标签)。
- **LDA**: 有监督, 找**最能分开类别**的方向(用标签)。

分类前降维时, LDA 投影通常比 PCA 更利于区分类别。


In [ ]:
from sklearn.decomposition import PCA
X_lda = LinearDiscriminantAnalysis(n_components=2).fit_transform(Xtr, y_tr)  # 3类→最多2维
X_pca = PCA(n_components=2).fit_transform(Xtr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, data, name in [(axes[0], X_lda, "LDA 投影(有监督, 类分得开)"),
                       (axes[1], X_pca, "PCA 投影(无监督, 仅最大方差)")]:
    sc_ = ax.scatter(data[:,0], data[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=30)
    ax.set_title(name); ax.set_xlabel("分量1"); ax.set_ylabel("分量2")
plt.tight_layout(); plt.show()
print("LDA 投影里三类几乎完全分离(因为它用了标签优化可分性); PCA 只保方差")
print("LDA 降维上限 = 类别数-1 =", 3-1, "维")


<a id="5"></a>
## 5. 关系网: LDA / QDA / LR / NB / The Family Tree

四个生成/判别分类器的关系(面试爱串):

| 模型 | 类型 | 对 P(x|c) 的假设 | 边界 |
|---|---|---|---|
| **QDA** | 生成 | 各类高斯, 各自协方差 | 二次 |
| **LDA** | 生成 | 各类高斯, **共享**协方差 | 线性 |
| **Gaussian NB** | 生成 | 各类高斯, **对角**协方差(特征独立) | 二次(对角) |
| **逻辑回归** | 判别 | 不建模 P(x|c), 直接建 P(c|x) | 线性 |

- LDA 与逻辑回归**边界都线性**, 但 LDA 多了高斯假设: 假设成立时 LDA 更高效(小数据), 不成立时逻辑回归更稳健。
- Gaussian NB 是 QDA 限定协方差对角的特例。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
print("Wine 上四模型 5-fold CV 准确率:")
for name, m in [("QDA", QuadraticDiscriminantAnalysis()),
                ("LDA", LinearDiscriminantAnalysis()),
                ("GaussianNB", GaussianNB()),
                ("LogisticReg", LogisticRegression(max_iter=2000))]:
    print(f"  {name:<12} {cross_val_score(m, sc.transform(X), y, cv=5).mean():.3f}")
print("Wine 各类近高斯 → 判别分析表现优异, 与逻辑回归相当")


<a id="6"></a>
## 6. 小结 / Summary

```
LDA/QDA: 每类多元高斯 + 贝叶斯; 判别函数含马氏距离, 分到最近(先验加权)的类
QDA: 各类自有协方差 → 二次边界, 参数多易过拟合
LDA: 共享协方差 → 二次项消去 → 线性边界, 更稳(偏差换方差)
LDA 降维: 最大化 类间/类内 方差, 最多 K-1 维; 有监督(vs PCA 无监督)
关系: GaussianNB=对角协方差的QDA; LDA与逻辑回归边界都线性(高斯假设 vs 无假设)
```

### 💡 面试速查
1. **LDA 共享协方差→线性边界; QDA 各类协方差→二次边界**
2. **LDA vs PCA**: 有监督(最大可分) vs 无监督(最大方差); LDA 降维上限 K-1
3. **LDA vs 逻辑回归**: 都线性边界, LDA 多高斯假设(成立时小数据更优)
4. **Gaussian NB = 协方差对角的 QDA**
5. QDA 参数多, 数据少/类内样本少时易过拟合 → 退回 LDA

### 下一节
**5.13 多类与多标签**——系统讲 OvR/OvO 把二分类器拼成多分类, 以及"一个样本属多个类"的多标签问题(分类器链)。
